# Lesson 5 — Baseline and First Model

Read `README.md` (or `README.pl.md`) first. Before fitting anything clever, this notebook sets the bar a real model has to clear — then clears it.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from task import impute_driver_experience, load_shipments, split_shipments

df = load_shipments()
train_df, test_df = split_shipments(df)
train_df, test_df = impute_driver_experience(train_df, test_df)
len(train_df), len(test_df)

## Two naive guesses, scored in-sample first

"Always expect the historical average" vs. "always expect on time" — both computed from the training data only, so you can build intuition for MAE/RMSE before the test set ever enters the picture.

In [ ]:
from task import (
    mean_absolute_error,
    predict_mean_baseline,
    predict_zero_baseline,
    root_mean_squared_error,
)

train_actual = train_df["delay_minutes"]
mean_pred_train = predict_mean_baseline(train_df, train_df)
zero_pred_train = predict_zero_baseline(train_df)
mean_mae_train = mean_absolute_error(train_actual, mean_pred_train)
mean_rmse_train = root_mean_squared_error(train_actual, mean_pred_train)
zero_mae_train = mean_absolute_error(train_actual, zero_pred_train)
zero_rmse_train = root_mean_squared_error(train_actual, zero_pred_train)
print("mean-baseline  MAE:", mean_mae_train, " RMSE:", mean_rmse_train)
print("zero-baseline  MAE:", zero_mae_train, " RMSE:", zero_rmse_train)

## Fit the model on train, score everything on test

Now bring in `test_df` — for the first and only time in this lesson — to find out whether the baseline, and a real model, actually generalize. The baseline here uses the *training* mean, never the test mean.

In [ ]:
from task import FEATURE_COLUMNS, fit_model, predict_delay

model = fit_model(train_df)
test_actual = test_df["delay_minutes"]
model_pred = predict_delay(model, test_df)
mean_pred_test = predict_mean_baseline(train_df, test_df)
zero_pred_test = predict_zero_baseline(test_df)

predictions = {
    "zero-baseline": zero_pred_test,
    "mean-baseline": mean_pred_test,
    "linear model": model_pred,
}
for name, pred in predictions.items():
    mae = mean_absolute_error(test_actual, pred)
    rmse = root_mean_squared_error(test_actual, pred)
    print(name, " MAE:", mae, " RMSE:", rmse)

## What did the model learn?

The sign of each coefficient — does it match what Lesson 4's correlations suggested? Raw coefficient size isn't directly comparable across these features (a coefficient in minutes-per-stop isn't on the same scale as one in minutes-per-year) — Lesson 8's exemplar explains a fairer way to compare them.

In [ ]:
dict(zip(FEATURE_COLUMNS, model.coef_, strict=True))

## Your notes

By how much (in minutes of MAE) does the model beat the fair, held-out mean-baseline? And: `weather` had a real effect in Lesson 4 but isn't in `FEATURE_COLUMNS` here — what would you need to do to add it?